<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/supervised/02_multiple_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S2 複数の入力の線形回帰と最小二乗法(notebook)

本文:[`02-multiple-linear-regression.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/supervised/02-multiple-linear-regression.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

W_TRUE = np.array([2.0, 0.3, 1.5])     # 本当の重み:切片 2 %、距離 1 m あたり 0.3 %、重さ 1 kg あたり 1.5 %

def make_data(N, rng, corr=0.0, sigma=1.0):
    # 距離 x1 [m](0〜50)と荷物の重さ x2 [kg](およそ 0〜5)から、バッテリーの消費量 y [%] を作る
    # corr を大きくすると、遠くへ行く走行ほど重い荷物を運ぶ(x1 と x2 に関係ができる)
    x1 = rng.uniform(0, 50, N)
    x2 = corr * x1 / 10 + (1 - corr) * rng.uniform(0, 5, N) + rng.normal(0, 0.5, N)
    X = np.column_stack([x1, x2])                     # 形 (N, 2):行がサンプル、列が入力
    y = W_TRUE[0] + X @ W_TRUE[1:] + rng.normal(0, sigma, N)
    return X, y

## 1. データ行列

**予想**:50 回の走行のデータで、切片の列を足したデータ行列 $X$ の形と、 $X^\top X$ の形はそれぞれ何か(本文の確認課題2)。

- 予想:(ここに書く)

**実装**:入力の配列(形 `(N, D)`)の左に、すべて 1 の列を足す関数 `add_bias` を書く(本文 §1)。

In [ ]:
def add_bias(X):
    raise NotImplementedError("TODO: 1列目に、すべて 1 の列を足す(np.column_stack と np.ones を使う)")

rng = np.random.default_rng(0)
X, y = make_data(50, rng)
A = add_bias(X)
check("データ行列の形は (50, 3)", A.shape, (50, 3))
check("1列目はすべて 1", A[:, 0], np.ones(50))
check("2・3列目は入力そのまま", A[:, 1:], X)

## 2. 最小二乗法で重みを求める

**手計算**:本文の確認課題1(4つの点に平面を当てはめる)を解く。

- 手計算:(ここに書く)

**実装**:正規方程式 (1) で重みを求める関数 `fit` を書く。予測の関数 `predict` は用意してある。

In [ ]:
def fit(X, y):
    A = add_bias(X)
    raise NotImplementedError("TODO: 正規方程式 A^T A w = A^T y を np.linalg.solve で解く(inv は使わない)")

def predict(w, X):
    return add_bias(X) @ w

X4 = np.array([[0, 0], [1, 0], [0, 1], [1, 1]], dtype=float)
y4 = np.array([1, 3, 0, 2], dtype=float)
check("4つの点:w = (1, 2, -1)", fit(X4, y4), [1, 2, -1])
w = fit(X, y)
check("lstsq と同じ答え", w, np.linalg.lstsq(add_bias(X), y, rcond=None)[0])
print("推定した重み:", np.round(w, 3), "  本当の重み:", W_TRUE)

## 3. 残差と決定係数

**実装**:決定係数 $R^2$ を返す関数 `r2_score` を書く(本文 §4)。あわせて、残差が $X$ の列と直交すること、残差の平均が 0 になることを確かめる。

In [ ]:
def r2_score(y, y_hat):
    raise NotImplementedError("TODO: 1 - (残差の2乗の和)/(y と平均の差の2乗の和)")

e = y - predict(w, X)
check("残差はデータ行列のすべての列と直交する(A^T e = 0)", add_bias(X).T @ e, np.zeros(3), tol=1e-6)
check("残差の平均は 0", e.mean(), 0.0, tol=1e-9)
check("4つの点は平面にのるので R^2 = 1", r2_score(y4, predict(fit(X4, y4), X4)), 1.0)
check("平均で予測すると R^2 = 0", r2_score(y, np.full_like(y, y.mean())), 0.0)
print(f"R^2 = {r2_score(y, predict(w, X)):.3f}")

In [ ]:
y_hat = predict(w, X)
plt.figure(figsize=(4, 4))
plt.plot(y, y_hat, "o", alpha=0.6)
lims = [y.min(), y.max()]
plt.plot(lims, lims, "k--", label="y_hat = y")
plt.xlabel("measured y [%]"); plt.ylabel("predicted y_hat [%]"); plt.legend(); plt.show()

点が破線(予測 = 実際)のまわりに集まっていれば、よく当てはまっている。破線からのずれが残差である。

## 4. 重みの意味

遠くへ行く走行ほど重い荷物を運ぶデータ(`corr = 0.8`)を作る。

**予想**:距離だけで回帰した傾き(S1 の方法)と、距離と重さの両方で回帰した $w_1$ は、どちらが大きいか。本当の値は $w_1 = 0.3$ である(本文 §3、確認課題3)。

- 予想:(ここに書く)

In [ ]:
rng = np.random.default_rng(1)
Xc, yc = make_data(5000, rng, corr=0.8)
print("距離と重さの相関係数:", round(np.corrcoef(Xc[:, 0], Xc[:, 1])[0, 1], 2))
w_both = fit(Xc, yc)
w_dist = fit(Xc[:, :1], yc)                        # 距離だけを入力にする
print("距離と重さの両方:w =", np.round(w_both, 3))
print("距離だけ        :w =", np.round(w_dist, 3))
check("両方を入れると、距離の重みは本当の値 0.3 に近い", w_both[1], 0.3, tol=0.01)
check("距離だけだと、傾きは 0.3 + 1.5 × 0.08 = 0.42 に近い", w_dist[1], 0.42, tol=0.01)

距離だけで回帰すると、傾きは約 0.42 になり、本当の 0.3 より大きい。遠くへ行く走行ほど荷物が重いので、「荷物が重いことによる消費」(重さ 1 kg あたり 1.5 %)が、距離の傾きに混ざるからである。このデータでは、距離が 10 m 増えると重さが平均 0.8 kg 増えるので、混ざる分は $1.5 \times 0.08 = 0.12$ である。

## 5. 入力の単位と標準化

**予想**:荷物の重さを kg ではなく g で記録すると、 $w_2$ と予測はそれぞれどうなるか(本文の確認課題4)。

- 予想:(ここに書く)

In [ ]:
X_g = X.copy()
X_g[:, 1] *= 1000                                  # 重さを g にする
w_g = fit(X_g, y)
print("kg のとき w =", np.round(w, 4), "  g のとき w =", np.round(w_g, 7))
check("w2 は 1/1000 になる", w_g[2], w[2] / 1000, tol=1e-9)
check("予測は変わらない", predict(w_g, X_g), predict(w, X), tol=1e-6)
A, A_g = add_bias(X), add_bias(X_g)
print(f"条件数 cond(X^T X):kg のとき {np.linalg.cond(A.T @ A):.1e}、g のとき {np.linalg.cond(A_g.T @ A_g):.1e}")

単位を変えると重みは変わるが、予測は変わらない。重みの大きさだけでは、入力の影響の大きさは比べられない。また、単位がそろっていないと条件数が大きくなる(計算の誤差が大きくなりやすい)。

**実装**:各入力を標準化する関数 `standardize` を書く(本文 §5)。標準化した入力で回帰し、予測が変わらないことを確かめる。

**予想**:標準化したとき、距離と重さのどちらの重みが大きいか。

In [ ]:
def standardize(X):
    raise NotImplementedError("TODO: 列ごとに平均 mean と標準偏差 std を求め、(X - mean) / std と mean・std を返す")

Z, mean, std = standardize(X)
check("標準化した入力の平均は 0", Z.mean(axis=0), [0, 0], tol=1e-9)
check("標準化した入力の標準偏差は 1", Z.std(axis=0), [1, 1], tol=1e-9)
w_z = fit(Z, y)
check("標準化しても予測は変わらない", predict(w_z, Z), predict(w, X), tol=1e-9)
print("標準化した重み(1標準偏差あたりの変化):距離", round(w_z[1], 2), "%、重さ", round(w_z[2], 2), "%")
print("入力の標準偏差:距離", np.round(std[0], 1), "m、重さ", np.round(std[1], 2), "kg")

重さ 1 kg あたりの重み(1.5)は、距離 1 m あたりの重み(0.3)より大きい。しかし、このデータでは距離のほうが大きく変わる(標準偏差 約 14 m)ので、1標準偏差あたりでは、距離の影響のほうが大きい。

## 6. 似た入力が2つあるとき

距離を、もう1つのセンサーでも測ったとする。2つ目の値は、1つ目とほとんど同じ(誤差の標準偏差 0.5 m)である。

**予想**:似た入力を足すと、距離の重みと予測は、データを作り直すたびにどれくらいばらつくか(本文 §6)。

- 予想:(ここに書く)

In [ ]:
def add_similar(X, rng, noise=0.5):
    # 距離 x1 を、別のセンサーで測った値 x3 を足す
    return np.column_stack([X, X[:, 0] + rng.normal(0, noise, len(X))])

rng = np.random.default_rng(2)
X_test, y_test = make_data(1000, rng)
X_test3 = add_similar(X_test, rng)
w_2, w_3, mse_2, mse_3 = [], [], [], []
for seed in range(300):
    r = np.random.default_rng(100 + seed)
    Xs, ys = make_data(50, r)
    Xs3 = add_similar(Xs, r)
    w_2.append(fit(Xs, ys)); w_3.append(fit(Xs3, ys))
    mse_2.append(np.mean((y_test - predict(w_2[-1], X_test)) ** 2))
    mse_3.append(np.mean((y_test - predict(w_3[-1], X_test3)) ** 2))
w_2, w_3 = np.array(w_2), np.array(w_3)
print(f"入力2つ:距離の重み 平均 {w_2[:, 1].mean():.3f}、標準偏差 {w_2[:, 1].std():.3f}")
print(f"入力3つ:距離の重み 平均 {w_3[:, 1].mean():.3f}、標準偏差 {w_3[:, 1].std():.3f}   "
      f"2つ目のセンサーの重み 標準偏差 {w_3[:, 3].std():.3f}")
print(f"入力3つ:2つの距離の重みの和 平均 {(w_3[:, 1] + w_3[:, 3]).mean():.3f}、標準偏差 {(w_3[:, 1] + w_3[:, 3]).std():.3f}")
print(f"評価用のデータの平均二乗誤差:入力2つ {np.mean(mse_2):.3f}、入力3つ {np.mean(mse_3):.3f}")
A3 = add_bias(Xs3)
print(f"条件数 cond(X^T X):入力2つ {np.linalg.cond(add_bias(Xs).T @ add_bias(Xs)):.1e}、入力3つ {np.linalg.cond(A3.T @ A3):.1e}")

似た入力を足すと、それぞれの距離の重みは大きくばらつく(片方が大きな正、もう片方が負、のようになることもある)。しかし、2つの重みの**和**のばらつきは小さく、予測の誤差もほとんど変わらない。データからは「距離の影響の合計」は決まるが、それを2つのセンサーにどう分けるかは決まらないからである。

## 7. 入力を増やしすぎると

学習用のデータを 40 個にし、関係のない入力(でたらめな数)を 0 個から 34 個まで足していく。

**予想**:学習用のデータでの誤差と、評価用のデータでの誤差は、それぞれどう変わるか(本文 §7)。

- 予想:(ここに書く)

In [ ]:
def add_noise_inputs(X, k, rng):
    return np.column_stack([X, rng.normal(0, 1, (len(X), k))]) if k > 0 else X

ks = list(range(0, 35, 2))
train_err, test_err = [], []
for k in ks:
    tr, te = [], []
    for seed in range(50):
        r = np.random.default_rng(1000 + seed)
        Xs, ys = make_data(40, r)
        Xt, yt = make_data(1000, r)
        Xs_k, Xt_k = add_noise_inputs(Xs, k, r), add_noise_inputs(Xt, k, r)
        w_k = fit(Xs_k, ys)
        tr.append(np.mean((ys - predict(w_k, Xs_k)) ** 2))
        te.append(np.mean((yt - predict(w_k, Xt_k)) ** 2))
    train_err.append(np.mean(tr)); test_err.append(np.mean(te))

plt.figure(figsize=(5, 3.2))
plt.semilogy(ks, train_err, "o-", label="training data")
plt.semilogy(ks, test_err, "s-", label="test data")
plt.axhline(1.0, color="k", ls="--", label="noise variance (sigma^2 = 1)")
plt.xlabel("number of unrelated inputs added"); plt.ylabel("mean squared error"); plt.legend(); plt.show()

学習用の誤差は、入力を足すほど小さくなる。足した入力で、たまたまの雑音まで説明してしまうからである。一方、評価用の誤差は大きくなり、入力の数(重みの数)が学習用のデータの数 40 に近づくと、急に大きくなる。これが**過学習**である(S4 で詳しく学ぶ)。学習用のデータの誤差や $R^2$ だけで、入力を足すかどうかを決めてはいけない。

## 8. 条件を変える

1. 学習用のデータの数 $N$ を 10、100、1000 にして、距離の重み $w_1$ のばらつき(300 個の seed での標準偏差)がどう変わるか
2. 雑音をなくす(`sigma=0`)と、推定した重みはどうなるか
3. §6 の2つ目のセンサーの誤差を 0.05 m と 5 m にすると、重みのばらつきはどう変わるか

In [ ]:
# 1〜3 をここで試す

## 9. 説明(自分の言葉で書く)

1. 複数の入力の回帰で、重み $w_1$ は何を表すか。距離だけで回帰した傾きと違うのはなぜか
   - (ここに書く)
2. 入力の単位を変えると、重みと予測はそれぞれどうなるか。入力の影響を比べたいときは、どうすればよいか
   - (ここに書く)
3. 学習用のデータの誤差だけで、入力を足すかどうかを決めてはいけないのはなぜか
   - (ここに書く)